# 02. Pareto Efficiency Frontier & 12-Month Inference TCO Model
**Objective:** Evaluate **Capability-per-Dollar** and **Capability-per-Ton-CO2** across frontier models, and model full lifecycle TCO (Pre-Training CapEx/OpEx + 12-Month Serving Inference).

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/02_gold_exports/models_enriched_gold.csv')
pareto_df = df.dropna(subset=['Composite_Capability_Index', 'total_cost_usd']).copy()
pareto_df = pareto_df[(pareto_df['Composite_Capability_Index'] > 0) & (pareto_df['total_cost_usd'] > 0)]

# Compute Pareto Frontier (Minimize Cost, Maximize Composite Capability Index)
pareto_df = pareto_df.sort_values('total_cost_usd')
frontier_rows = []
max_cap = -1.0
for _, row in pareto_df.iterrows():
    if row['Composite_Capability_Index'] > max_cap:
        frontier_rows.append(row)
        max_cap = row['Composite_Capability_Index']
frontier = pd.DataFrame(frontier_rows)
print(f'Models evaluated: {len(pareto_df)} | Pareto-optimal models: {len(frontier)}')
frontier[['model_name', 'organization', 'publication_year', 'total_cost_usd', 'Composite_Capability_Index', 'usd_per_capability_point', 'co2_tons_per_capability_point']]

Models evaluated: 17 | Pareto-optimal models: 6


,model_name,organization,publication_year,total_cost_usd,Composite_Capability_Index,usd_per_capability_point,co2_tons_per_capability_point
384,Llama 2-7B,Meta AI,2023,1.142594e+05,45.800000,2494.75,NaN
421,Falcon-40B,Technology Innovation Institute,2023,3.197832e+05,56.900000,5620.09,4.2796
426,LLaMA-65B,Meta AI,2023,5.780263e+05,63.400000,9117.13,5.6036
383,Llama 2-70B,Meta AI,2023,1.102561e+06,69.900000,15773.41,8.5492
181,DeepSeek-R1 (May 2025),DeepSeek,2025,6.770000e+06,76.325758,88698.76,NaN
277,Llama 3.1-405B,Meta AI,2024,5.288543e+07,84.400000,626604.67,249.5789


In [2]:
# Chart 9: Frontier Capability vs. Training Cost Pareto Curve
fig = px.scatter(
    pareto_df, x='total_cost_usd', y='Composite_Capability_Index',
    color='organization', hover_name='model_name',
    size=pareto_df['facility_total_energy_mwh'].fillna(pareto_df['facility_total_energy_mwh'].median()), log_x=True,
    title='AI Economics Pareto Frontier: Capability Index vs. Training Cost (USD)',
    labels={'total_cost_usd': 'Estimated Training Cost (USD, Log Scale)', 'Composite_Capability_Index': 'Composite Frontier Capability Index (0-100)'},
    template='plotly_white', width=1050, height=580
)
fig.add_trace(go.Scatter(
    x=frontier['total_cost_usd'], y=frontier['Composite_Capability_Index'],
    mode='lines+markers+text', text=frontier['model_name'], textposition='top left',
    line=dict(color='#DC2626', width=2.5, dash='dash'),
    marker=dict(size=10, color='#DC2626', symbol='diamond'),
    name='Pareto Efficiency Frontier'
))
fig.write_image('../assets/charts/chart9_pareto_frontier.png', scale=2)
fig.show()

In [3]:
# 12-Month Lifecycle TCO Simulator (Pre-Training + Production Inference Serving)
def simulate_12m_tco(df_models, daily_requests=10_000_000, avg_output_tokens=500, watts_per_1k_tokens=0.45):
    sim = df_models.dropna(subset=['total_cost_usd', 'facility_total_energy_mwh']).copy()
    annual_tokens_k = (daily_requests * avg_output_tokens * 365) / 1000.0
    # Scale inference energy modestly with model parameter count (normalized to 70B baseline)
    param_factor = np.log10(sim['parameters'].fillna(7e10).clip(lower=1e9)) / np.log10(7e10)
    sim['annual_inference_mwh'] = (annual_tokens_k * watts_per_1k_tokens * param_factor) / 1e6 * sim['facility_pue']
    sim['annual_inference_electricity_usd'] = sim['annual_inference_mwh'] * sim['tariff_usd_mwh']
    sim['annual_inference_co2_tons'] = (sim['annual_inference_mwh'] * 1000.0 * sim['carbon_gco2_kwh']) / 1e6
    sim['inference_to_training_energy_ratio'] = sim['annual_inference_mwh'] / sim['facility_total_energy_mwh']
    return sim[['model_name', 'organization', 'facility_total_energy_mwh', 'annual_inference_mwh', 'inference_to_training_energy_ratio', 'total_cost_usd', 'annual_inference_electricity_usd']]

tco_summary = simulate_12m_tco(df)
tco_summary.sort_values('facility_total_energy_mwh', ascending=False).head(10)

,model_name,organization,facility_total_energy_mwh,annual_inference_mwh,inference_to_training_energy_ratio,total_cost_usd,annual_inference_electricity_usd
209,Grok 3,xAI,327734.95,1133.802844,0.003460,2.178355e+08,92971.833172
331,Gemini 1.0 Ultra,Google DeepMind,127259.96,985.500000,0.007744,3.071942e+07,80811.000000
277,Llama 3.1-405B,Meta AI,66871.30,1054.775797,0.015773,5.288543e+07,86491.615329
420,GPT-4 (Mar 2023),OpenAI,62752.96,1113.643326,0.017746,3.733430e+07,91318.752745
287,Nemotron-4 340B,Nvidia,25778.13,1047.871806,0.040650,2.127102e+07,85925.488117
375,Falcon-180B,Technology Innovation Institute,19404.86,1022.772778,0.052707,1.074350e+07,122732.733401
370,Amazon Titan,Amazon,17375.01,1026.930787,0.059104,7.933465e+06,84208.324499
501,PaLM (540B),Google Research,8899.33,1066.154619,0.119802,3.060365e+06,87424.678737
661,OpenAI Five,OpenAI,7709.45,817.834930,0.106082,4.328311e+06,67062.464287
311,MegaScale (Production),"ByteDance,Peking University",7442.64,1171.930508,0.157462,2.614019e+06,89066.718616
